In [6]:
import os
import pandas as pd
import numpy as np
database = pd.read_excel("data/DB_SEB_VD.ods",sheet_name="DB_SEB_VD",header=0,engine='odf')
database = database.filter(['clip','timestamp'])
database.dropna(subset=["clip"],inplace=True)

database["time"] = database["timestamp"].apply(lambda x: pd.to_datetime(x,format='mixed',dayfirst=True))
database = database.filter(['clip','time'])




KeyError: "None of ['clip'] are in the columns"

In [15]:
data_path = "resultats"
l = []
for indiv in os.listdir(data_path)[:3]:
    #Add date to dataframe
    indiv_path = os.path.join(data_path,indiv,'metadata.csv')
    indiv_dataframe = pd.read_csv(indiv_path)
    indiv_dataframe['name'] = indiv
    indiv_dataframe = indiv_dataframe.filter(['name','video','crop_path'])
    indiv_dataframe = indiv_dataframe.set_index('video').join(database.set_index('clip'),how='left')
    indiv
    #Add entry to dataframe (one entry = all videos taken at the same spot within a 2 hours range)
    # A revoir pour la nuit?
    indiv_dataframe['clip'] = indiv_dataframe.index
    indiv_dataframe= indiv_dataframe.sort_values("time").reset_index(drop=True)
    times = indiv_dataframe["time"].to_numpy()
    entry = np.zeros(len(times), dtype=np.int32)
    group = 0
    start = times[0]
    for i in range(1, len(times)):
        if times[i] - start >= np.timedelta64(2, 'h'):
            group += 1
            start = times[i]
        entry[i] = group
    indiv_dataframe["entry"] = entry
    l.append(indiv_dataframe)
    indiv_dataframe = indiv_dataframe.filter(['clip','entry']).drop_duplicates()
    metadata = pd.read_csv(indiv_path).set_index('clip')
    metadata = metadata.join(indiv_dataframe.set_index('video'),how='left')
    metadata['video'] = metadata.index
    metadata.to_csv(indiv_path,index=False)
data = pd.concat(l)
data.to_csv(os.path.join(data_path,'data_summary.csv'),index=False)

data = data.filter(['name','entry'])
data.groupby(['name','entry']).count()

KeyError: "None of ['clip'] are in the columns"

In [16]:
indiv_path = os.path.join(data_path,indiv,'metadata.csv')
indiv_dataframe = pd.read_csv(indiv_path)
indiv_dataframe['name'] = indiv
indiv_dataframe = indiv_dataframe.filter(['name','video','crop_path'])
indiv_dataframe = indiv_dataframe.set_index('video').join(database.set_index('clip'),how='left')
indiv
#Add entry to dataframe (one entry = all videos taken at the same spot within a 2 hours range)
# A revoir pour la nuit?
indiv_dataframe['clip'] = indiv_dataframe.index
indiv_dataframe= indiv_dataframe.sort_values("time").reset_index(drop=True)
times = indiv_dataframe["time"].to_numpy()
entry = np.zeros(len(times), dtype=np.int32)
group = 0
start = times[0]
for i in range(1, len(times)):
    if times[i] - start >= np.timedelta64(2, 'h'):
        group += 1
        start = times[i]
    entry[i] = group
indiv_dataframe["entry"] = entry
l.append(indiv_dataframe)

In [17]:
indiv_dataframe

,name,crop_path,time,clip,entry
0,Ayo,resultats/Ayo/crops/id_0003/frame_000255_id_00...,2025-09-16 19:20:08,SEB_180362.AVI,0
1,Ayo,resultats/Ayo/crops/id_0003/frame_000765_id_00...,2025-09-16 19:20:08,SEB_180362.AVI,0
2,Ayo,resultats/Ayo/crops/id_0003/frame_000780_id_00...,2025-09-16 19:20:08,SEB_180362.AVI,0
3,Ayo,resultats/Ayo/crops/id_0003/frame_000795_id_00...,2025-09-16 19:20:08,SEB_180362.AVI,0
4,Ayo,resultats/Ayo/crops/id_0003/frame_000810_id_00...,2025-09-16 19:20:08,SEB_180362.AVI,0
...,...,...,...,...,...
287,Ayo,resultats/Ayo/crops/id_0005/frame_000615_id_00...,2025-10-02 06:37:08,SEB_182294.AVI,2
288,Ayo,resultats/Ayo/crops/id_0005/frame_000630_id_00...,2025-10-02 06:37:08,SEB_182294.AVI,2
289,Ayo,resultats/Ayo/crops/id_0005/frame_000690_id_00...,2025-10-02 06:37:08,SEB_182294.AVI,2
290,Ayo,resultats/Ayo/crops/id_0005/frame_000705_id_00...,2025-10-02 06:37:08,SEB_182294.AVI,2


In [18]:
indiv_dataframe = indiv_dataframe.filter(['clip','entry']).drop_duplicates()